# Kalibrering af pulsoximeter med Braun-reference

Denne notebook dokumenterer den praktiske kalibrering af det byggede pulsoximeter mod et kommercielt Braun-pulsoximeter. Formålet er at finde en brugbar kalibreringskonstant til demo og sammenligning under normale måleforhold.

Vigtigt: Braun-målingen i denne test ligger konstant ved 98 % SpO2. Derfor kan målingen ikke alene bestemme både hældning (`a`) og offset (`b`) i den lineære model. Med kun ét SpO2-niveau kan man enten:

- fastholde `a` fra en kendt/litteraturbaseret model og beregne `b`, eller
- bruge flere referencepunkter med forskellige SpO2-niveauer for at fitte både `a` og `b`.

I denne notebook fastholdes `a = -24.87`, og `b` bestemmes ud fra den målte middelværdi af R.

## Model

Pulsoximeteret beregner først forholdstallet

$$R = \frac{AC_{660}/DC_{660}}{AC_{940}/DC_{940}}$$

Derefter omsættes R til SpO2 med en lineær model:

$$SpO_2 = a \cdot R + b$$

I den nuværende måling var Braun-referenceværdien konstant 98 %. Hvis `a` fastholdes, kan `b` beregnes som:

$$b = SpO_{2,reference} - a \cdot R_{middel}$$

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
metrics_path = Path("../data/calibration/braun_normal_01_metrics.csv")
if not metrics_path.exists():
    metrics_path = Path("data/calibration/braun_normal_01_metrics.csv")

metrics = pd.read_csv(metrics_path)
metrics.head()

## Manuel aflæsning fra Braun-pulsoximeter

Braun-pulsoximeteret blev placeret på en anden finger end det byggede pulsoximeter. Begge blev filmet samtidig, og Braun-displayet blev aflæst hvert 5. sekund. Tiderne nedenfor er angivet relativt til start af målingen i Python-scriptet.

In [ ]:
braun = pd.DataFrame(
    [
        (0, 98, 51, "Start; Braun havde målt ca. 30 s før Python-målingen"),
        (5, 98, 49, ""),
        (10, 98, 48, ""),
        (15, 98, 48, ""),
        (20, 98, 48, ""),
        (25, 98, 51, ""),
        (30, 98, 53, ""),
        (35, 98, 53, ""),
        (40, 98, 52, ""),
        (45, 98, 50, ""),
        (50, 98, 49, ""),
        (55, 98, 49, ""),
        (60, 98, 49, ""),
        (65, 98, 48, ""),
        (70, 98, 48, ""),
        (75, 98, 48, ""),
        (80, 98, 49, ""),
        (85, 98, 49, ""),
        (90, 98, 50, ""),
    ],
    columns=["time_s", "Braun_SpO2", "Braun_HR", "note"],
)

braun

## Sammenkobling af Braun-aflæsninger og egne målinger

Python-scriptet gemmer mange målepunkter i CSV-filen. Braun-displayet opdaterer langsommere og er aflæst manuelt. Derfor sammenlignes hver Braun-aflæsning med middelværdien fra det byggede pulsoximeter i et vindue på ±2.5 s omkring samme tidspunkt.

In [ ]:
def mean_in_window(df, time_s, column, half_window_s=2.5):
    window = df[(df["t_s"] >= time_s - half_window_s) & (df["t_s"] <= time_s + half_window_s)]
    values = window[column].dropna()
    return float(values.mean()) if len(values) else np.nan


paired = braun.copy()
paired["R_egen"] = [mean_in_window(metrics, t, "R") for t in paired["time_s"]]
paired["SpO2_egen_foer_kalibrering"] = [mean_in_window(metrics, t, "spo2") for t in paired["time_s"]]
paired["HR_egen"] = [mean_in_window(metrics, t, "bpm") for t in paired["time_s"]]
paired["HR_fejl"] = paired["HR_egen"] - paired["Braun_HR"]

paired

## Valg af stabilt område

De første sekunder kan påvirkes af fingerplacering, filterhistorik og den tid det tager før pulssignalet bliver stabilt. Derfor bruges målepunkter efter 30 s til kalibreringen. Dette kan ændres i variablen `stabil_start_s`.

In [ ]:
stabil_start_s = 30
a_fast = -24.87

stable = paired[(paired["time_s"] >= stabil_start_s) & paired["R_egen"].notna()].copy()

R_middel = stable["R_egen"].mean()
R_std = stable["R_egen"].std(ddof=1)
Braun_SpO2_middel = stable["Braun_SpO2"].mean()
Braun_HR_middel = stable["Braun_HR"].mean()
HR_egen_middel = stable["HR_egen"].mean()
HR_abs_fejl_middel = stable["HR_fejl"].abs().mean()

b_kalibreret = Braun_SpO2_middel - a_fast * R_middel

resultat = pd.DataFrame(
    {
        "parameter": [
            "stabil_start_s",
            "a_fast",
            "R_middel",
            "R_std",
            "Braun_SpO2_middel",
            "b_kalibreret",
            "Braun_HR_middel",
            "HR_egen_middel",
            "middel_abs_HR_fejl",
        ],
        "værdi": [
            stabil_start_s,
            a_fast,
            R_middel,
            R_std,
            Braun_SpO2_middel,
            b_kalibreret,
            Braun_HR_middel,
            HR_egen_middel,
            HR_abs_fejl_middel,
        ],
    }
)

resultat

Den kalibrerede model fra denne måling er derfor:

$$SpO_2 = -24.87 \cdot R + b_{kalibreret}$$

hvor `b_kalibreret` beregnes i cellen ovenfor. For denne måling ligger værdien omkring 138.3, afhængigt af præcis hvor det stabile område vælges.

Denne kalibrering betyder ikke, at systemet er medicinsk valideret over hele SpO2-området. Den viser, at systemets visning kan bringes i overensstemmelse med Braun-referenceinstrumentet ved normal iltmætning omkring 98 %.

In [ ]:
metrics["spo2_kalibreret"] = a_fast * metrics["R"] + b_kalibreret

fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True, constrained_layout=True)

axes[0].plot(metrics["t_s"], metrics["R"], label="R fra eget pulsoximeter")
axes[0].axvline(stabil_start_s, color="black", linestyle="--", linewidth=1, label="Start på stabilt område")
axes[0].set_ylabel("R")
axes[0].grid(True, alpha=0.3)
axes[0].legend(frameon=False)

axes[1].plot(metrics["t_s"], metrics["spo2"], label="SpO2 før kalibrering")
axes[1].plot(metrics["t_s"], metrics["spo2_kalibreret"], label="SpO2 efter kalibrering")
axes[1].scatter(paired["time_s"], paired["Braun_SpO2"], color="black", s=25, label="Braun-aflæsninger")
axes[1].set_ylabel("SpO2 [%]")
axes[1].set_ylim(65, 105)
axes[1].grid(True, alpha=0.3)
axes[1].legend(frameon=False)

axes[2].plot(metrics["t_s"], metrics["bpm"], label="HR fra eget pulsoximeter")
axes[2].scatter(paired["time_s"], paired["Braun_HR"], color="black", s=25, label="Braun HR")
axes[2].set_xlabel("Tid [s]")
axes[2].set_ylabel("HR [BPM]")
axes[2].grid(True, alpha=0.3)
axes[2].legend(frameon=False)

plt.show()

## Hvorfor kan både `a` og `b` ikke bestemmes her?

Hvis alle referencepunkter har samme SpO2-værdi, ligger de på en vandret linje. Det giver ikke nok information om hældningen mellem R og SpO2. Mange forskellige kombinationer af `a` og `b` kan ramme 98 % ved den målte R-værdi.

For at bestemme både `a` og `b` kræves mindst to forskellige SpO2-referencepunkter, og i praksis gerne flere. Hvis man senere har flere referencepunkter, kan de fittes med lineær regression som vist nedenfor.

In [ ]:
# Eksempel til fremtidig brug, hvis der senere findes flere referencepunkter
# med forskellige SpO2-værdier. Tabellen nedenfor bruger den nuværende måling,
# hvor alle Braun-værdier er 98 %, og er derfor IKKE nok til en robust fitning
# af både a og b.

fit_data = stable[["R_egen", "Braun_SpO2"]].dropna()

if fit_data["Braun_SpO2"].nunique() >= 2:
    a_fit, b_fit = np.polyfit(fit_data["R_egen"], fit_data["Braun_SpO2"], deg=1)
    print(f"Fit med flere SpO2-niveauer: a = {a_fit:.3f}, b = {b_fit:.3f}")
else:
    print("Alle Braun-SpO2-værdier er ens i denne måling.")
    print("Derfor fastholdes a, og kun b beregnes:")
    print(f"a = {a_fast:.3f}")
    print(f"b = {b_kalibreret:.3f}")

## Kommando til efterfølgende demo

Efter denne kalibrering kan scriptet køres med den beregnede offset-værdi. Et eksempel til en kort eksamensdemo er:

```bash
python live_pulsoximeter_måling.py \
  --start-delay 15 \
  --duration 60 \
  --warmup 10 \
  --spo2-a -24.87 \
  --spo2-b 138.3
```

I en rapport eller eksamen bør resultatet beskrives som en praktisk kalibrering mod et Braun-forbruger-pulsoximeter ved normal iltmætning, ikke som en klinisk validering.